## TECH CHALLENGE - FASE 3: Exploração do Dataset MedQuAD

Este notebook tem como objetivo explorar a estrutura do dataset MedQuAD, entender o formato dos arquivos XML e preparar os dados para uso no
assistente médico que foi requisitado para fase 3.

Nesta etapa, serão realizadas as seguintes atividades:

- localização dos arquivos do dataset no projeto
- inspeção inicial da estrutura dos arquivos XML
- identificação dos campos de pergunta e resposta
- extração estruturada dos dados
- preparação de um arquivo processado para uso posterior em RAG e fine-tuning

O foco inicial será a coleção `1_CancerGov_QA`, por estar mais alinhada
ao tema já trabalhado nas fases anteriores do projeto.

In [1]:
from pathlib import Path
import xml.etree.ElementTree as ET
import json
import pandas as pd

## 1. Definição do caminho do dataset

Nesta etapa, será definido o caminho da pasta onde foi armazenado o
dataset MedQuAD dentro do projeto.

Inicialmente, a exploração será feita na coleção `1_CancerGov_QA`,
pois ela contém perguntas e respostas relacionadas ao domínio oncológico,
o que torna essa base mais aderente ao contexto do projeto.

In [2]:
base_path = Path("../data/medical_qa/raw/MedQuAD/1_CancerGov_QA")
files = list(base_path.glob("*.xml"))

print(f"Total de arquivos XML encontrados: {len(files)}")
print("\nPrimeiros 8 arquivos:")
for f in files[:8]:
    print(f.name)

Total de arquivos XML encontrados: 116

Primeiros 8 arquivos:
0000001_1.xml
0000001_2.xml
0000001_3.xml
0000001_4.xml
0000001_5.xml
0000001_6.xml
0000001_7.xml
0000003_1.xml


## 2. Inspeção inicial de um arquivo XML

Um arquivo será selecionado para inspeção manual.
Essa etapa é importante para compreender a estrutura bruta do XML e verificar como as informações estão organizadas internamente.

In [3]:
sample_file = files[0]
sample_file

WindowsPath('../data/medical_qa/raw/MedQuAD/1_CancerGov_QA/0000001_1.xml')

In [4]:
with open(sample_file, "r", encoding="utf-8") as f: content = f.read()

print(content[:2000])

<?xml version="1.0" encoding="UTF-8"?>
<Document id="0000001_1" source="CancerGov" url="https://www.cancer.gov/types/leukemia/patient/adult-all-treatment-pdq">
<Focus>Adult Acute Lymphoblastic Leukemia</Focus>
<FocusAnnotations>
	<UMLS>
		<CUIs>
			<CUI>C0751606</CUI>
		</CUIs>
		<SemanticTypes>
			<SemanticType>T191</SemanticType>
		</SemanticTypes>
		<SemanticGroup>Disorders</SemanticGroup>
	</UMLS>
</FocusAnnotations>
<QAPairs>
	<QAPair pid="1">
			<Question qid="0000001_1-1" qtype="information">What is (are) Adult Acute Lymphoblastic Leukemia ?</Question>
			<Answer>Key Points
                    - Adult acute lymphoblastic leukemia (ALL) is a type of cancer in which the bone marrow makes too many lymphocytes (a type of white blood cell).    - Leukemia may affect red blood cells, white blood cells, and platelets.    - Previous chemotherapy and exposure to radiation may increase the risk of developing ALL.    - Signs and symptoms of adult ALL include fever, feeling tired, and easy b

## 3. Leitura da estrutura XML

Após observar o conteúdo bruto do arquivo, será feito o parsing do XML
com a biblioteca `xml.etree.ElementTree`.

O objetivo desta etapa é identificar a tag raiz e compreender a organização
hierárquica dos elementos presentes no arquivo.

In [5]:
tree = ET.parse(sample_file)
root = tree.getroot()

print("Tag raiz:", root.tag)
print("\nFilhos imediatos da raiz:")
for child in root:
    print(child.tag)

Tag raiz: Document

Filhos imediatos da raiz:
Focus
FocusAnnotations
QAPairs


## 4. Identificação das tags existentes

Nesta etapa, serão listadas as tags encontradas no arquivo XML para
identificar onde estão armazenadas as perguntas e as respostas.

Esse passo é importante porque permite construir um parser adequado
para a estrutura específica do MedQuAD.

In [6]:
tags = set()

for elem in root.iter():
    tags.add(elem.tag)

print("Tags encontradas no XML:")
for tag in sorted(tags):
    print(tag)

Tags encontradas no XML:
Answer
CUI
CUIs
Document
Focus
FocusAnnotations
QAPair
QAPairs
Question
SemanticGroup
SemanticType
SemanticTypes
UMLS


## 5. Inspeção dos elementos textuais

Agora serão exibidos os elementos do XML que possuem conteúdo textual.
Essa inspeção ajuda a localizar os campos que efetivamente armazenam
as perguntas e respostas do dataset.

In [7]:
for elem in root.iter():
    if elem.text and elem.text.strip():
        print(f"{elem.tag} => {elem.text.strip()[:300]}")
        print("-" * 80)

Focus => Adult Acute Lymphoblastic Leukemia
--------------------------------------------------------------------------------
CUI => C0751606
--------------------------------------------------------------------------------
SemanticType => T191
--------------------------------------------------------------------------------
SemanticGroup => Disorders
--------------------------------------------------------------------------------
Question => What is (are) Adult Acute Lymphoblastic Leukemia ?
--------------------------------------------------------------------------------
Answer => Key Points
                    - Adult acute lymphoblastic leukemia (ALL) is a type of cancer in which the bone marrow makes too many lymphocytes (a type of white blood cell).    - Leukemia may affect red blood cells, white blood cells, and platelets.    - Previous chemotherapy and exposure to radia
--------------------------------------------------------------------------------
Question => What are the symptom

## 6. Criação da função de extração de pares pergunta-resposta

Com base na estrutura observada no XML, será criada uma função para extrair os pares de pergunta e resposta de cada arquivo.

Essa função permitirá automatizar a leitura de múltiplos arquivos da coleção selecionada.

In [8]:
def parse_medquad_xml(sample_filesample_file):
    tree = ET.parse(sample_file)
    root = tree.getroot()

    records = []
    file_name = Path(sample_file).name

    for idx, qa_pair in enumerate(root.findall(".//QAPair")):
        question = qa_pair.findtext("Question")
        answer = qa_pair.findtext("Answer")

        if question and answer:
            records.append({
                "id": f"{file_name}_{idx}",
                "source_file": file_name,
                "collection": "CancerGov",
                "question": question.strip(),
                "answer": answer.strip()
            })

    return records

## 7. Teste da função em um arquivo de exemplo

Nesta etapa, a função criada será aplicada a um único arquivo XML, com o objetivo de validar se a extração dos pares pergunta-resposta está ocorrendo corretamente.

In [9]:
records = parse_medquad_xml(sample_file)

print(f"Total de pares extraídos do arquivo de exemplo: {len(records)}")
records[:2]

Total de pares extraídos do arquivo de exemplo: 7


[{'id': '0000001_1.xml_0',
  'source_file': '0000001_1.xml',
  'collection': 'CancerGov',
  'question': 'What is (are) Adult Acute Lymphoblastic Leukemia ?',
  'answer': 'Key Points\n                    - Adult acute lymphoblastic leukemia (ALL) is a type of cancer in which the bone marrow makes too many lymphocytes (a type of white blood cell).    - Leukemia may affect red blood cells, white blood cells, and platelets.    - Previous chemotherapy and exposure to radiation may increase the risk of developing ALL.    - Signs and symptoms of adult ALL include fever, feeling tired, and easy bruising or bleeding.     - Tests that examine the blood and bone marrow are used to detect (find) and diagnose adult ALL.    - Certain factors affect prognosis (chance of recovery) and treatment options.\n                \n                \n                    Adult acute lymphoblastic leukemia (ALL) is a type of cancer in which the bone marrow makes too many lymphocytes (a type of white blood cell).\n

## 8. Extração dos dados de todos os arquivos da coleção

Após validar a função em um arquivo de exemplo, será feita a extração dos dados de todos os arquivos XML da coleção `1_CancerGov_QA`.

Ao final, será gerada uma lista consolidada com todos os pares pergunta-resposta encontrados.

In [10]:
all_records = []

for file_path in files:
    try:
        extracted = parse_medquad_xml(file_path)
        all_records.extend(extracted)
    except Exception as e:
        print(f"Erro ao processar {file_path.name}: {e}")

print(f"Total de registros extraídos: {len(all_records)}")

Total de registros extraídos: 812


## 9. Conversão para DataFrame

Nesta etapa, os registros extraídos serão convertidos para um DataFrame,
facilitando a visualização, inspeção e futuras etapas de pré-processamento.

In [11]:
df_medquad = pd.DataFrame(all_records)

print(df_medquad.shape)
df_medquad.head()

(812, 5)


,id,source_file,collection,question,answer
0,0000001_1.xml_0,0000001_1.xml,CancerGov,What is (are) Adult Acute Lymphoblastic Leukem...,Key Points\n - Adult acute ...
1,0000001_1.xml_1,0000001_1.xml,CancerGov,What are the symptoms of Adult Acute Lymphobla...,"Signs and symptoms of adult ALL include fever,..."
2,0000001_1.xml_2,0000001_1.xml,CancerGov,How to diagnose Adult Acute Lymphoblastic Leuk...,Tests that examine the blood and bone marrow a...
3,0000001_1.xml_3,0000001_1.xml,CancerGov,What is the outlook for Adult Acute Lymphoblas...,Certain factors affect prognosis (chance of re...
4,0000001_1.xml_4,0000001_1.xml,CancerGov,Who is at risk for Adult Acute Lymphoblastic L...,Previous chemotherapy and exposure to radiatio...


## 10. Verificação de valores ausentes e duplicados

Antes de salvar o dataset processado, é importante realizar uma verificação
simples de qualidade dos dados, observando a presença de valores nulos
e registros duplicados.

In [12]:
print("Valores nulos:")
print(df_medquad.isnull().sum())

print("\nRegistros duplicados:")
print(df_medquad.duplicated().sum())

Valores nulos:
id             0
source_file    0
collection     0
question       0
answer         0
dtype: int64

Registros duplicados:
805


## 11. Salvamento do dataset processado

Por fim, o conjunto de dados tratado será salvo na pasta `processed`,
permitindo seu reaproveitamento nas próximas fases do pipeline,
como construção da base vetorial, RAG e fine-tuning.

In [13]:
output_path = Path("../data/medical_qa/processed/medquad_cancergov_qa.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_records, f, ensure_ascii=False, indent=2)

print(f"Arquivo salvo em: {output_path.resolve()}")

Arquivo salvo em: C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\medical_qa\processed\medquad_cancergov_qa.json


## 12. Visualização de exemplos finais

Para concluir esta etapa, serão exibidos alguns exemplos do dataset processado, permitindo validar visualmente o resultado da extração.

In [14]:
for i, row in df_medquad.head(3).iterrows():
    print(f"Registro {i+1}")
    print(f"Arquivo de origem: {row['source_file']}")
    print(f"Pergunta: {row['question']}")
    print(f"Resposta: {row['answer'][:500]}")
    print("=" * 100)

Registro 1
Arquivo de origem: 0000001_1.xml
Pergunta: What is (are) Adult Acute Lymphoblastic Leukemia ?
Resposta: Key Points
                    - Adult acute lymphoblastic leukemia (ALL) is a type of cancer in which the bone marrow makes too many lymphocytes (a type of white blood cell).    - Leukemia may affect red blood cells, white blood cells, and platelets.    - Previous chemotherapy and exposure to radiation may increase the risk of developing ALL.    - Signs and symptoms of adult ALL include fever, feeling tired, and easy bruising or bleeding.     - Tests that examine the blood and bone marrow are u
Registro 2
Arquivo de origem: 0000001_1.xml
Pergunta: What are the symptoms of Adult Acute Lymphoblastic Leukemia ?
Resposta: Signs and symptoms of adult ALL include fever, feeling tired, and easy bruising or bleeding. The early signs and symptoms of ALL may be like the flu or other common diseases. Check with your doctor if you have any of the following:         - Weakness or feel

## 13. Preparação de documentos para RAG

Nesta etapa os pares pergunta-resposta serão convertidos em documentos
que poderão ser indexados em uma base vetorial.

Cada documento conterá:
- pergunta
- resposta
- metadados da fonte

In [15]:
from langchain_core.documents import Document

documents = []

for _, row in df_medquad.iterrows():

    text = f"""
Question: {row['question']}
Answer: {row['answer']}
"""

    metadata = {
        "source_file": row["source_file"],
        "collection": row["collection"],
        "id": row["id"]
    }

    documents.append(
        Document(
            page_content=text,
            metadata=metadata
        )
    )

print("Total de documentos criados:", len(documents))
documents[0]

Total de documentos criados: 812


Document(metadata={'source_file': '0000001_1.xml', 'collection': 'CancerGov', 'id': '0000001_1.xml_0'}, page_content='\nQuestion: What is (are) Adult Acute Lymphoblastic Leukemia ?\nAnswer: Key Points\n                    - Adult acute lymphoblastic leukemia (ALL) is a type of cancer in which the bone marrow makes too many lymphocytes (a type of white blood cell).    - Leukemia may affect red blood cells, white blood cells, and platelets.    - Previous chemotherapy and exposure to radiation may increase the risk of developing ALL.    - Signs and symptoms of adult ALL include fever, feeling tired, and easy bruising or bleeding.     - Tests that examine the blood and bone marrow are used to detect (find) and diagnose adult ALL.    - Certain factors affect prognosis (chance of recovery) and treatment options.\n                \n                \n                    Adult acute lymphoblastic leukemia (ALL) is a type of cancer in which the bone marrow makes too many lymphocytes (a type of w

## 14. Criação de Chunks para processamento da base vetorial

In [16]:
import sys
!{sys.executable} -m pip install -U langchain-text-splitters


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 15. Criação da base vetorial

Para permitir recuperação semântica das informações médicas,
os documentos serão convertidos em embeddings e armazenados
em uma base vetorial utilizando FAISS.

In [18]:
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

embedding_model = OllamaEmbeddings(model="nomic-embed-text")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

split_documents = text_splitter.split_documents(documents)

print("Documentos originais:", len(documents))
print("Chunks gerados:", len(split_documents))

vector_store = FAISS.from_documents(
    split_documents,
    embedding_model
)

print("Base vetorial criada com sucesso.")

Documentos originais: 812
Chunks gerados: 8700
Base vetorial criada com sucesso.


O uso de chunking foi necessário para evitar que textos maiores excedam o limite de entrada do modelo de embeddings.
A estratégia escolhida foi o RecursiveCharacterTextSplitter, recomendado pelo LangChain para textos genéricos, com sobreposição entre chunks para
preservar contexto semântico.

In [20]:
faiss_path = "../data/vector_store"

vector_store.save_local(faiss_path)

print("Índice salvo em:", faiss_path)

Índice salvo em: ../data/vector_store


## 16. Teste de recuperação semântica

Após a criação da base vetorial, será testado o mecanismo de recuperação
para verificar se a busca semântica retorna trechos relevantes do dataset.

In [21]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

query = "What are symptoms of breast cancer?"

docs = retriever.invoke(query)

for d in docs:
    print("----")
    print(d.page_content[:400])
    print(d.metadata)

----
Question: What are the symptoms of Adult Acute Lymphoblastic Leukemia ?
{'source_file': '0000001_1.xml', 'collection': 'CancerGov', 'id': '0000001_1.xml_1'}
----
Question: What are the symptoms of Adult Acute Lymphoblastic Leukemia ?
{'source_file': '0000001_1.xml', 'collection': 'CancerGov', 'id': '0000001_1.xml_1'}
----
Question: What are the symptoms of Adult Acute Lymphoblastic Leukemia ?
{'source_file': '0000001_1.xml', 'collection': 'CancerGov', 'id': '0000001_1.xml_1'}


## 17. Teste de geração com LLM local (Ollama)

Nesta etapa, os documentos recuperados pelo retriever serão utilizados como contexto para um modelo de linguagem local executado via Ollama.

O objetivo é gerar respostas médicas em linguagem natural utilizando apenas o conteúdo recuperado da base de conhecimento.

In [23]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="mistral",
    temperature=0.2
)

In [26]:
context = "\n\n".join([doc.page_content for doc in docs])

prompt = f"""
You are a medical educational assistant focused on breast cancer information.

Use only the context below to answer the question.
Do not invent information.
If the context is insufficient, clearly say that the available sources are insufficient.
Do not provide a definitive diagnosis.
Do not prescribe treatment.

Context:
{context}

Question:
{query}
"""

In [27]:
response = llm.invoke(prompt)
print(response.content)

 The context provided does not offer information about the symptoms of Adult Acute Lymphoblastic Leukemia (ALL). However, it does provide a question about the symptoms of breast cancer.

Symptoms of breast cancer may include:
1. A new lump or mass in the breast or underarm area that is often hard, irregularly shaped, and may feel fixed (not movable).
2. Changes in the size, shape, or appearance of the breast.
3. Changes to the nipple, such as retracting (pulling inward), redness, or scaling, or discharge that is not milk.
4. Swelling or lumps in the armpit.
5. Pain in the breast or nipple area.
6. An unusual thickening in or near the breast, including the nipple.
7. Changes in the skin of the breast, such as dimpling, puckering, or redness.

Please consult a healthcare professional for any concerns about potential symptoms.


A resposta gerada nesta etapa já representa um fluxo básico de RAG
(Retrieval-Augmented Generation), no qual o modelo de linguagem utiliza
os documentos recuperados da base vetorial como contexto para formular
a resposta.

## 18. Explainability: indicação das fontes consultadas

Para aumentar a rastreabilidade da resposta gerada, o sistema apresentará as fontes dos documentos recuperados pelo mecanismo de RAG.
Essa estratégia contribui para explainability, permitindo identificar quais registros do dataset foram utilizados como base para a resposta.

In [28]:
def format_sources(docs):
    lines = []

    for i, doc in enumerate(docs, start=1):
        lines.append(
            f"[Source {i}] "
            f"dataset=MedQuAD | "
            f"collection={doc.metadata.get('collection')} | "
            f"source_file={doc.metadata.get('source_file')} | "
            f"id={doc.metadata.get('id')}"
        )

    return "\n".join(lines)

In [30]:
sources_text = format_sources(docs)

final_answer = f"""
{response.content}

Sources consulted:
{sources_text}
""".strip()

print(final_answer)

The context provided does not offer information about the symptoms of Adult Acute Lymphoblastic Leukemia (ALL). However, it does provide a question about the symptoms of breast cancer.

Symptoms of breast cancer may include:
1. A new lump or mass in the breast or underarm area that is often hard, irregularly shaped, and may feel fixed (not movable).
2. Changes in the size, shape, or appearance of the breast.
3. Changes to the nipple, such as retracting (pulling inward), redness, or scaling, or discharge that is not milk.
4. Swelling or lumps in the armpit.
5. Pain in the breast or nipple area.
6. An unusual thickening in or near the breast, including the nipple.
7. Changes in the skin of the breast, such as dimpling, puckering, or redness.

Please consult a healthcare professional for any concerns about potential symptoms.

Sources consulted:
[Source 1] dataset=MedQuAD | collection=CancerGov | source_file=0000001_1.xml | id=0000001_1.xml_1
[Source 2] dataset=MedQuAD | collection=Cancer

Nesta implementação, a LLM é responsável apenas pela geração textual da resposta. Já a listagem das fontes é montada programaticamente em Python, evitando que o modelo invente referências inexistentes.
Essa separação melhora a confiabilidade da saída final.

## 19. Criação de uma função de consulta ao assistente médico

Para facilitar testes e futuras integrações com a estrutura modular do projeto,
será criada uma função que executa o pipeline completo:

1. recebe a pergunta do usuário
2. recupera documentos relevantes
3. monta o contexto
4. gera a resposta com a LLM
5. exibe as fontes consultadas

In [31]:
def ask_medical_assistant(question, retriever, llm, k=3):
    docs = retriever.invoke(question)

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
You are a medical educational assistant focused on breast cancer information.

Use only the context below to answer the question.
Do not invent information.
If the context is insufficient, clearly say that the available sources are insufficient.
Do not provide a definitive diagnosis.
Do not prescribe treatment.
Always answer in a clear and educational tone.

Context:
{context}

Question:
{question}
"""

    response = llm.invoke(prompt)

    final_response = f"""
{response.content}

Sources consulted:
{format_sources(docs)}
""".strip()

    return final_response

In [32]:
question_test = "What are the symptoms of breast cancer?"
print(ask_medical_assistant(question_test, retriever, llm))

The context provided does not contain information about the symptoms of breast cancer. However, I can share that breast cancer symptoms can include:

1. A new lump in the breast or underarm that feels different from the surrounding tissue.
2. Change in size or shape of the breast.
3. Nipple discharge other than breast milk, especially if it's bloody.
4. Change in the skin texture of the breast or nipple, such as dimpling, puckering, or orchestration.
5. An inverted nipple (pulling inward instead of sticking out).
6. Scaly, red, or swollen skin on the breast, nipple, or areola.
7. Pain in the breast or nipple area.

If you suspect you or someone else might have breast cancer, it's important to consult with a healthcare professional for a proper diagnosis and guidance on next steps.

Regarding Adult Acute Lymphoblastic Leukemia, the symptoms can include fatigue, frequent infections, easy bruising or bleeding, weight loss, swollen lymph nodes, and frequent fevers. However, these symptoms 

In [33]:
## 20. Avaliação qualitativa inicial do pipeline RAG

Nesta etapa, o sistema já é capaz de:

- recuperar trechos semanticamente relevantes do MedQuAD
- utilizar esses trechos como contexto para a geração da resposta
- apresentar as fontes consultadas
- operar localmente via Ollama, sem dependência de APIs pagas

Esse resultado representa a primeira versão funcional do assistente médico
virtual proposto para a Fase 3.

SyntaxError: invalid syntax (1816609234.py, line 3)